# Testes da Camada de Persistência

**Aluno:** João Victor Lemes Faria — 202302614
**Projeto:** Controle Financeiro Pessoal — ORMLite + SQLite

Create e Read das entidades mapeadas, exercitando as três cardinalidades do modelo:
**1:N** (`Usuario` → `Conta` → `Transacao`), **1:1** (`Categoria` ↔ `Orcamento`) e
**N:M** (`Transacao` ↔ `Categoria`, via `TransacaoCategoria`).

Pré-requisito: projeto compilado em `target/classes` (a extensão Java do VS Code faz isso
automaticamente; com Maven, `mvn compile`).

In [9]:
%maven com.j256.ormlite:ormlite-jdbc:6.1
%maven org.xerial:sqlite-jdbc:3.47.1.0

In [10]:
// Os dois caminhos cobrem o Jupyter aberto na raiz do projeto ou nesta pasta.
%classpath target/classes
%classpath ../../../../../target/classes

In [11]:
import com.controlefinanceiro.config.DatabaseHelper;
import com.controlefinanceiro.dao.*;
import com.controlefinanceiro.model.*;

import java.math.BigDecimal;
import java.nio.file.Files;
import java.nio.file.Path;
import java.text.SimpleDateFormat;
import java.util.Date;

SimpleDateFormat FMT = new SimpleDateFormat("dd/MM/yyyy");

## 1. Conexão e criação das tabelas

In [12]:
Files.deleteIfExists(Path.of("financeiro.db"));

DatabaseHelper db = new DatabaseHelper();

UsuarioDao usuarioDao = db.getUsuarioDao();
ContaDao contaDao = db.getContaDao();
CategoriaDao categoriaDao = db.getCategoriaDao();
TransacaoDao transacaoDao = db.getTransacaoDao();
OrcamentoDao orcamentoDao = db.getOrcamentoDao();
TransacaoCategoriaDao transacaoCategoriaDao = db.getTransacaoCategoriaDao();

## 2. Create

Um registro de cada entidade. Os ids são gerados pelo banco (`generatedId = true`) e
escritos de volta no objeto após o `create`.

In [13]:
Usuario usuario = new Usuario("João Victor", "joao@email.com", "senha123");
usuarioDao.create(usuario);

Conta conta = new Conta("Conta Corrente", "CORRENTE", new BigDecimal("2500.00"), usuario);
contaDao.create(conta);

Categoria categoria = new Categoria("Alimentação", "DESPESA");
categoriaDao.create(categoria);

Orcamento orcamento = new Orcamento(new BigDecimal("900.00"), "2026-09", usuario, categoria);
orcamentoDao.create(orcamento);

Transacao transacao = new Transacao("Supermercado", new BigDecimal("430.75"),
                                    FMT.parse("08/09/2026"), "DESPESA", conta);
transacaoDao.create(transacao);

TransacaoCategoria rateio = new TransacaoCategoria(new BigDecimal("430.75"), transacao, categoria);
transacaoCategoriaDao.create(rateio);

System.out.println("ids gerados:");
System.out.println("  usuario            = " + usuario.getId());
System.out.println("  conta              = " + conta.getId());
System.out.println("  categoria          = " + categoria.getId());
System.out.println("  orcamento          = " + orcamento.getId());
System.out.println("  transacao          = " + transacao.getId());
System.out.println("  transacaoCategoria = " + rateio.getId());

ids gerados:
  usuario            = 1
  conta              = 1
  categoria          = 1
  orcamento          = 1
  transacao          = 1
  transacaoCategoria = 1


## 3. Read

Cada registro é relido do banco por `queryForId`. Os objetos das chaves estrangeiras vêm
preenchidos por causa de `foreignAutoRefresh = true`, o que confirma as associações.

In [14]:
Usuario u = usuarioDao.queryForId(usuario.getId());
System.out.println("Usuario   : " + u.getNome() + " <" + u.getEmail() + ">");

Conta c = contaDao.queryForId(conta.getId());
System.out.println("Conta     : " + c.getNome() + " | saldo R$ " + c.getSaldo()
    + " | usuario: " + c.getUsuario().getNome());

Categoria cat = categoriaDao.queryForId(categoria.getId());
System.out.println("Categoria : " + cat.getNome() + " (" + cat.getTipo() + ")");

Orcamento o = orcamentoDao.queryForId(orcamento.getId());
System.out.println("Orcamento : limite R$ " + o.getValorLimite() + " em " + o.getMesReferencia()
    + " | categoria: " + o.getCategoria().getNome()
    + " | usuario: " + o.getUsuario().getNome());

Transacao t = transacaoDao.queryForId(transacao.getId());
System.out.println("Transacao : " + t.getDescricao() + " | R$ " + t.getValor()
    + " | " + FMT.format(t.getData()) + " | conta: " + t.getConta().getNome());

TransacaoCategoria tc = transacaoCategoriaDao.queryForId(rateio.getId());
System.out.println("Rateio    : R$ " + tc.getValorRateado()
    + " | transacao: " + tc.getTransacao().getDescricao()
    + " | categoria: " + tc.getCategoria().getNome());

Usuario   : João Victor <joao@email.com>
Conta     : Conta Corrente | saldo R$ 2500.00 | usuario: João Victor
Categoria : Alimentação (DESPESA)
Orcamento : limite R$ 900.00 em 2026-09 | categoria: Alimentação | usuario: João Victor
Transacao : Supermercado | R$ 430.75 | 08/09/2026 | conta: Conta Corrente
Rateio    : R$ 430.75 | transacao: Supermercado | categoria: Alimentação


In [ ]:
System.out.println("registros por tabela:");
System.out.println("  usuarios             = " + usuarioDao.countOf());
System.out.println("  contas               = " + contaDao.countOf());
System.out.println("  categorias           = " + categoriaDao.countOf());
System.out.println("  orcamentos           = " + orcamentoDao.countOf());
System.out.println("  transacoes           = " + transacaoDao.countOf());
System.out.println("  transacao_categorias = " + transacaoCategoriaDao.countOf());

db.fechar();

registros por tabela:
  usuarios             = 1
  contas               = 1
  categorias           = 1
  orcamentos           = 1
  transacoes           = 1
  transacao_categorias = 1

Conexão encerrada.
